In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt


In [3]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize
x_train, x_test = x_train / 255.0, x_test / 255.0

# One-hot encoding
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)


In [4]:
def build_small_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

small_cnn = build_small_cnn()
small_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_small = small_cnn.fit(x_train, y_train_cat, epochs=10, validation_split=0.2)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.3691 - loss: 1.7259 - val_accuracy: 0.5572 - val_loss: 1.2554
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5851 - loss: 1.1756 - val_accuracy: 0.5997 - val_loss: 1.1353
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6457 - loss: 1.0202 - val_accuracy: 0.6491 - val_loss: 1.0116
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6846 - loss: 0.9057 - val_accuracy: 0.6362 - val_loss: 1.0438
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7138 - loss: 0.8290 - val_accuracy: 0.6752 - val_loss: 0.9492
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7414 - loss: 0.7548 - val_accuracy: 0.6732 - val_loss: 0.9610
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7556 - loss: 0.6916 - val_accuracy: 0.6854 - val_loss: 0.9445
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7770 - loss: 0.6375 -

In [5]:
def fine_tune_model(base_model_class):
    base_model = base_model_class(include_top=False, input_shape=(32,32,3), weights='imagenet')

    for layer in base_model.layers[:-2]:
        layer.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train_cat, epochs=10, validation_split=0.2)
    return model

vgg_model = fine_tune_model(VGG16)
resnet_model = fine_tune_model(ResNet50)


Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - accuracy: 0.3634 - loss: 1.8624 - val_accuracy: 0.5493 - val_loss: 1.3343
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.5708 - loss: 1.2742 - val_accuracy: 0.5945 - val_loss: 1.1927
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.6106 - loss: 1.1516 - val_accuracy: 0.6141 - val_loss: 1.1304
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.6297 - loss: 1.0900 - val_accuracy: 0.6242 - val_loss: 1.0908
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.6473 - loss: 1.0389 - val_accuracy: 0.6373 - val_loss: 1.0568
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.6596 - loss: 0.9991 - val_accuracy: 0.6421 - val_loss: 1.0330
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.6685 - loss: 0.9714 - val_accuracy: 0.6480 - val_loss: 1.0189
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.6800 -

In [6]:
def distill(student, teacher, temperature=3, alpha=0.5, epochs=10):
    soft_loss_fn = tf.keras.losses.KLDivergence()
    hard_loss_fn = tf.keras.losses.CategoricalCrossentropy()
    optimizer = tf.keras.optimizers.Adam()

    train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train_cat)).batch(64)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        for step, (images, labels) in enumerate(train_dataset):
            with tf.GradientTape() as tape:
                student_preds = student(images, training=True)
                teacher_preds = teacher(images, training=False)

                soft_logits = tf.nn.softmax(teacher_preds / temperature)
                student_soft = tf.nn.softmax(student_preds / temperature)

                loss_soft = soft_loss_fn(soft_logits, student_soft)
                loss_hard = hard_loss_fn(labels, student_preds)
                loss = alpha * loss_hard + (1 - alpha) * loss_soft

            grads = tape.gradient(loss, student.trainable_variables)
            optimizer.apply_gradients(zip(grads, student.trainable_variables))

    return student


In [7]:
# Clone small CNN architecture for distillation
small_student_1 = build_small_cnn()
small_student_1 = distill(small_student_1, vgg_model)


Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10


In [8]:
def distill_from_ensemble(student, teacher1, teacher2, temperature=3, alpha=0.5, epochs=10):
    soft_loss_fn = tf.keras.losses.KLDivergence()
    hard_loss_fn = tf.keras.losses.CategoricalCrossentropy()
    optimizer = tf.keras.optimizers.Adam()

    train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train_cat)).batch(64)

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        for step, (images, labels) in enumerate(train_dataset):
            with tf.GradientTape() as tape:
                student_preds = student(images, training=True)
                t1 = tf.nn.softmax(teacher1(images, training=False) / temperature)
                t2 = tf.nn.softmax(teacher2(images, training=False) / temperature)
                ensemble_logits = (t1 + t2) / 2.0
                student_soft = tf.nn.softmax(student_preds / temperature)

                loss_soft = soft_loss_fn(ensemble_logits, student_soft)
                loss_hard = hard_loss_fn(labels, student_preds)
                loss = alpha * loss_hard + (1 - alpha) * loss_soft

            grads = tape.gradient(loss, student.trainable_variables)
            optimizer.apply_gradients(zip(grads, student.trainable_variables))

    return student


In [9]:
# Clone small CNN again
small_student_2 = build_small_cnn()
small_student_2 = distill_from_ensemble(small_student_2, vgg_model, resnet_model)


Epoch 1/10
Epoch 2/10
Epoch 3/10
Epoch 4/10
Epoch 5/10
Epoch 6/10
Epoch 7/10
Epoch 8/10
Epoch 9/10
Epoch 10/10


In [30]:
# Compile for evaluation
small_cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
small_student_1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
small_student_2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Evaluate models
print("\nSmall CNN (original):")
small_cnn.evaluate(x_test, y_test_cat)

print("\nSmall CNN (distilled from VGG16):")
small_student_1.evaluate(x_test, y_test_cat)

print("\nSmall CNN (distilled from VGG16 + ResNet50 ensemble):")
small_student_2.evaluate(x_test, y_test_cat)



Small CNN (original):
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6755 - loss: 1.0191

Small CNN (distilled from VGG16):
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6950 - loss: 0.9376

Small CNN (distilled from VGG16 + ResNet50 ensemble):
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6827 - loss: 0.9466


[0.9570325016975403, 0.6805999875068665]